# 🎬 NTV Transcoder — Colab

Give it **public video links** → it downloads → transcodes to multi-resolution **HLS** (GPU/NVENC) →
uploads to **Cloudflare R2** → optionally **pushes the details to the NTV CMS**.

### Before you start
1. **Runtime → Change runtime type → GPU (T4 or L4)** — the custom `ffmpeg` needs an NVIDIA GPU.
2. Add your keys as **Colab Secrets** (🔑 in the left sidebar). Recommended names:
   - `GITHUB_TOKEN` — a GitHub PAT (read) so this private repo can be cloned
   - `CLOUDFLARE_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `BUCKET_NAME`
   - *(optional)* `CMS_URL`, `CMS_API_KEY`, `CDN_BASE_URL`
3. Run the cells top to bottom.


In [ ]:
#@title 🔧 1. Setup — clone repo, install custom ffmpeg + deps { display-mode: "form" }
#@markdown Needs a **GPU runtime**. For this **private** repo add a Colab secret `GITHUB_TOKEN`
#@markdown (or paste one below).
import os, sys, subprocess

REPO_SLUG           = "kailasa-ngpt/ntv-transcoder-colab" #@param {type:"string"}
BRANCH              = "main" #@param {type:"string"}
GITHUB_TOKEN_INLINE = "" #@param {type:"string"}

# --- GPU check ---
gpu = subprocess.run(["bash","-lc","nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU detected. Runtime -> Change runtime type -> GPU (T4/L4), then re-run.")
print("GPU:", gpu)

# --- token for private clone (inline field wins, else Colab secret) ---
token = GITHUB_TOKEN_INLINE.strip()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        token = ""

DEST = "/content/ntv-transcoder-colab"
clone_url = f"https://{token + '@' if token else ''}github.com/{REPO_SLUG}.git"
if not os.path.isdir(DEST):
    if subprocess.run(["git","clone","--depth","1","-b",BRANCH,clone_url,DEST]).returncode != 0:
        raise SystemExit("Clone failed. Private repo? Provide GITHUB_TOKEN (secret or field above).")
    # drop the token from the stored remote URL
    subprocess.run(["git","-C",DEST,"remote","set-url","origin",
                    f"https://github.com/{REPO_SLUG}.git"])
else:
    subprocess.run(["git","-C",DEST,"pull"])

os.chdir(DEST)
sys.path.insert(0, DEST)
print("Repo:", DEST)

# --- install the custom ffmpeg/ffprobe ---
subprocess.run(["bash","-lc",
    "sudo cp ffmpeg ffprobe /usr/local/bin/ && "
    "sudo chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe && hash -r"], check=True)

# --- python deps ---
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], check=True)

# --- verify GPU ffmpeg ---
enc = subprocess.run(["bash","-lc","ffmpeg -hide_banner -encoders | grep nvenc"],
                     capture_output=True, text=True).stdout.strip()
flt = subprocess.run(["bash","-lc","ffmpeg -hide_banner -filters | grep scale_npp"],
                     capture_output=True, text=True).stdout.strip()
print("\nnvenc encoders:\n", enc or "  (none)")
print("scale_npp filter:\n", flt or "  (none)")
print("\n" + ("Custom GPU ffmpeg ready." if (enc and flt)
             else "WARNING: NVENC/scale_npp missing - transcoding will fail. Check GPU runtime."))


In [ ]:
#@title 🔑 2. Credentials — Colab Secrets (recommended) or paste below { display-mode: "form" }
#@markdown Leave a field blank to fall back to the matching **Colab Secret**.
CLOUDFLARE_ACCOUNT_ID = "" #@param {type:"string"}
R2_ACCESS_KEY_ID      = "" #@param {type:"string"}
R2_SECRET_ACCESS_KEY  = "" #@param {type:"string"}
BUCKET_NAME           = "ntv-ott" #@param {type:"string"}
CDN_BASE_URL          = "" #@param {type:"string"}
#@markdown ---
#@markdown **NTV CMS push (optional)** — leave blank to skip pushing to NTV:
CMS_URL     = "" #@param {type:"string"}
CMS_API_KEY = "" #@param {type:"string"}

import os
def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

def _set(name, inline):
    val = (inline or "").strip() or _secret(name)
    if val:
        os.environ[name] = val
    return val

acc = _set("CLOUDFLARE_ACCOUNT_ID", CLOUDFLARE_ACCOUNT_ID)
ak  = _set("R2_ACCESS_KEY_ID",      R2_ACCESS_KEY_ID)
sk  = _set("R2_SECRET_ACCESS_KEY",  R2_SECRET_ACCESS_KEY)
bkt = _set("BUCKET_NAME",           BUCKET_NAME or "ntv-ott")
cdn = _set("CDN_BASE_URL",          CDN_BASE_URL)
cms_url = _set("CMS_URL",     CMS_URL)
cms_key = _set("CMS_API_KEY", CMS_API_KEY)

missing = [n for n, v in [("CLOUDFLARE_ACCOUNT_ID", acc), ("R2_ACCESS_KEY_ID", ak),
                          ("R2_SECRET_ACCESS_KEY", sk), ("BUCKET_NAME", bkt)] if not v]
print("Missing R2 creds:", ", ".join(missing) if missing else "none  (bucket: %s)" % bkt)
print("CDN base:", cdn or "(not set - URLs will use the R2 endpoint)")
print("NTV CMS push:", ("ENABLED -> " + cms_url) if (cms_url and cms_key) else "disabled")


In [ ]:
#@title 🎥 3. Videos to process — paste one link per line
#@markdown **Paste one video per line** in `LINKS`. Only the **link** is required.
#@markdown Add optional fields separated by ` | ` (pipes), in this order:
#@markdown
#@markdown &nbsp;&nbsp;`<drive_link> | <video_id> | <title> | <category> | <date YYYY-MM-DD>`
#@markdown
#@markdown - `video_id` (= R2 folder + CMS `videoId`) is auto-derived from the Drive file id if you leave it out.
#@markdown - `title` + `category` are only needed if you push to NTV (step 6).
LINKS = """
https://drive.google.com/file/d/FILE_ID_1/view?usp=sharing | vid-001 | Morning Satsang | Nithyananda Satsang 2025 | 2025-01-01
https://drive.google.com/file/d/FILE_ID_2/view?usp=sharing | vid-002 | Evening Satsang | Nithyananda Satsang 2025
https://drive.google.com/file/d/FILE_ID_3/view?usp=sharing
"""

import re, helpers

def _slug(s):
    return re.sub(r"[^A-Za-z0-9._-]+", "-", s.strip()).strip("-")[:64] or "video"

VIDEOS = []
for line in LINKS.strip().splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    parts = [p.strip() for p in line.split("|")]
    link = parts[0]
    vid  = parts[1] if len(parts) > 1 and parts[1] else (helpers.extract_drive_id(link) or _slug(link))
    VIDEOS.append({
        "link":        link,
        "video_id":    vid,
        "title":       parts[2] if len(parts) > 2 else "",
        "category":    parts[3] if len(parts) > 3 else "",
        "date":        parts[4] if len(parts) > 4 else "",
        "description": "",
    })

assert VIDEOS, "No links found — paste at least one link into LINKS."
print(f"{len(VIDEOS)} video(s) queued:")
for v in VIDEOS:
    print(f"  - {v['video_id']:<24} {v['link'][:60]}")


In [ ]:
#@title ⬇️ 4. Download source videos  (gdown for Drive, yt-dlp otherwise)
import os, helpers
os.makedirs("workspace", exist_ok=True)
for v in VIDEOS:
    try:
        v["local_path"] = helpers.download_video(v["link"], f"workspace/{v['video_id']}.mp4")
    except Exception as e:
        v["local_path"] = None
        print("DOWNLOAD FAILED", v["video_id"], "->", e)
print("\nDownloaded:", [v["video_id"] for v in VIDEOS if v.get("local_path")])


In [ ]:
#@title 🎞️ 5. Transcode → HLS (240p–1080p) → upload to R2  (+ thumbnail)
#@markdown The thumbnail is fetched **from YouTube via yt-dlp using `video_id`**
#@markdown (so set `video_id` to the YouTube id, even for Drive videos). If that
#@markdown fails it falls back to grabbing a frame from the downloaded video.
import sys, subprocess, helpers
RESOLUTIONS = ["1080p", "720p", "480p", "360p", "240p"]

for v in VIDEOS:
    vid, src = v["video_id"], v.get("local_path")
    print("=" * 64, "\nTranscoding:", vid)
    if not src:
        v["transcoded"] = False
        print("  skipped - no downloaded file"); continue

    rc = subprocess.run([sys.executable, "ott-transcoder.py",
                         "--video-path", src, "--video-id", vid])
    v["transcoded"] = (rc.returncode == 0)
    if not v["transcoded"]:
        print("  TRANSCODE FAILED:", vid); continue

    _, v["duration"] = helpers.get_duration(src)

    # Thumbnail: always try YouTube (yt-dlp + video_id) first, even for Drive
    # sources; fall back to a video frame if YouTube has none for this id.
    thumb = helpers.fetch_youtube_thumbnail(vid, f"workspace/{vid}_thumb.webp")
    if not thumb:
        print("  [thumb] youtube fetch failed; falling back to a video frame")
        thumb = helpers.make_thumbnail(src, f"workspace/{vid}_thumb.webp")
    if thumb:
        try:
            helpers.upload_file_to_r2(thumb, f"{vid}/thumbnail.webp", "image/webp")
            v["thumbnail"] = True
        except Exception as e:
            print("  thumbnail upload failed:", e)
    print("  done:", vid, "| duration:", v.get("duration"))

print("\nTranscoded:", [v["video_id"] for v in VIDEOS if v.get("transcoded")])


In [ ]:
#@title 📤 6. Push details to NTV CMS  (optional)
import os
cms_url, cms_key = os.environ.get("CMS_URL"), os.environ.get("CMS_API_KEY")
RESOLUTIONS = ["1080p", "720p", "480p", "360p", "240p"]

if not (cms_url and cms_key):
    print("Skipped - set CMS_URL + CMS_API_KEY in step 2 to enable.")
else:
    from ntv_cms import NTVClient
    client = NTVClient(cms_url, cms_key)
    for v in VIDEOS:
        vid = v["video_id"]
        if not v.get("transcoded"):
            print("skip (not transcoded):", vid); continue
        if not (v.get("title") and v.get("category")):
            print("skip (needs title + category):", vid); continue
        res = client.push_video(
            video_id=vid, title=v["title"], category=v["category"],
            resolutions=RESOLUTIONS, description=v.get("description"),
            duration=v.get("duration"), date=v.get("date"))
        print(("pushed " if res.get("success") else "FAILED "), vid, "->",
              res.get("cms_id") or res.get("error"))


In [ ]:
#@title ✅ 7. Summary — playback URLs
import os
base = (os.environ.get("CDN_BASE_URL") or "").rstrip("/")
if not base:
    base = ("https://%s.r2.cloudflarestorage.com/%s"
            % (os.environ.get("CLOUDFLARE_ACCOUNT_ID", "<account>"),
               os.environ.get("BUCKET_NAME", "ntv-ott")))
print("Base:", base, "\n")
for v in VIDEOS:
    vid = v["video_id"]
    print(("[ok] " if v.get("transcoded") else "[--] ") + vid)
    if v.get("transcoded"):
        print("    master:", f"{base}/{vid}/master.m3u8")
        print("    1080p :", f"{base}/{vid}/1080p/playlist.m3u8")
        if v.get("thumbnail"):
            print("    thumb :", f"{base}/{vid}/thumbnail.webp")


---
## 🔍 Optional: find a Drive file with rclone (only if you don't have a direct link)

Public links don't need rclone. Use this only to **search** a Drive/Team Drive for a file.
Upload your own `rclone.conf` (and service-account JSON if it references one) to `/content` first —
**neither is stored in the repo.**

```python
# !curl -s https://rclone.org/install.sh | sudo bash
# !rclone --config /content/rclone.conf ls gdrive: | grep -i "<search-term>"
# !rclone --config /content/rclone.conf copyto gdrive:Videos/<file>.mp4 workspace/<video_id>.mp4 --progress
```

Then set that `workspace/<video_id>.mp4` as the video's `local_path` and run steps 5–7.

See `docs/ffmpeg-build-steps.txt` for how the custom ffmpeg is built/loaded.
